Incerteza local das dez réplicas LRi–RF

In [ ]:
from pathlib import Path
import sys
import pandas as pd

sys.path.append("/code/scripts")

from analysis_config import ANALYSIS, RESULTS
from model_analysis_utils import (
    rf_model_configuration,
    rf_uncertainty_local_fast,
    rf_uncertainty_summary,
    rf_uncertainty_very_high_summary,
    very_high_threshold
)
from pilot_utils import clip_raster

In [ ]:
configurations = [
    {"area": "centro", "scenario": "C2", "pilot": "fundao"},
    {"area": "centro", "scenario": "C4", "pilot": "fundao"},
    {"area": "centro", "scenario": "C6", "pilot": "fundao"},
    {"area": "extremadura", "scenario": "C6", "pilot": "badajoz"}
]

CLASS_BREAKS = (
    ANALYSIS / "02_classes" /
    "class_breaks_and_inventory.xlsx"
)

PREDICT_BATCH_SIZE = 262144
N_JOBS = 2

## Produção dos rasters

Esta célula é a única parte pesada. Se os rasters de desvio-padrão e frequência já existirem, a configuração é reutilizada e os modelos não são reaplicados.

In [ ]:
threshold_rows = []

for config in configurations:
    area = config["area"]
    scenario = config["scenario"]
    pilot = config["pilot"]

    print("\n", "-" * 60)
    print("Configuração:", area, scenario, pilot)

    models_xlsx = (
        RESULTS / area / scenario /
        "lri_model" / "models.xlsx"
    )
    model_paths, feature_names, regional_features = (
        rf_model_configuration(models_xlsx)
    )

    municipality = (
        ANALYSIS / "02_classes" / pilot /
        f"municipality_{pilot}.gpkg"
    )
    out_dir = ANALYSIS / "06_rf_uncertainty" / pilot / scenario
    feature_dir = out_dir / "features"
    feature_dir.mkdir(parents=True, exist_ok=True)

    local_features = []

    for name, raster in zip(feature_names, regional_features):
        output_feature = feature_dir / name

        if not output_feature.exists() or output_feature.stat().st_size == 0:
            clip_raster(raster, municipality, output_feature)

        local_features.append(str(output_feature))

    map_id = f"{scenario}_susceptibility"
    threshold = very_high_threshold(
        CLASS_BREAKS,
        map_id,
        area=area
    )
    threshold_rows.append({
        "area": area,
        "scenario": scenario,
        "pilot": pilot,
        "map_id": map_id,
        "threshold_method": "regional_p80_ensemble_mean",
        "threshold": threshold
    })

    mean_regional = (
        RESULTS / area / scenario / "lri_model" /
        "class" / "rf_lri_base_mean_prob_1_10models.tif"
    )
    mean_local = out_dir / f"{scenario}_mean_{pilot}.tif"

    if not mean_local.exists() or mean_local.stat().st_size == 0:
        clip_raster(mean_regional, municipality, mean_local)

    sd_raster = out_dir / f"{scenario}_sd_{pilot}.tif"
    frequency_raster = (
        out_dir / f"{scenario}_very_high_frequency_{pilot}.tif"
    )

    completed = all(
        path.exists() and path.stat().st_size > 0
        for path in [sd_raster, frequency_raster]
    )

    if completed:
        print("Configuração já concluída; rasters reutilizados.")
    else:
        rf_uncertainty_local_fast(
            model_paths=model_paths,
            feature_rasters=local_features,
            threshold=threshold,
            output_sd=sd_raster,
            output_frequency=frequency_raster,
            block_size=PREDICT_BATCH_SIZE,
            n_jobs=N_JOBS
        )
        print("Configuração concluída:", area, scenario, pilot)

    stale_file = out_dir / "regional_thresholds.csv"
    stale_file.unlink(missing_ok=True)

In [ ]:
root = ANALYSIS / "06_rf_uncertainty"
thresholds_df = pd.DataFrame(threshold_rows)
local_tables = []
very_high_tables = []

for config in configurations:
    area = config["area"]
    scenario = config["scenario"]
    pilot = config["pilot"]
    out_dir = root / pilot / scenario

    sd_raster = out_dir / f"{scenario}_sd_{pilot}.tif"
    frequency_raster = (
        out_dir / f"{scenario}_very_high_frequency_{pilot}.tif"
    )
    class_raster = (
        ANALYSIS / "02_classes" / pilot / "rasters" /
        f"{scenario}_susceptibility_class_{pilot}.tif"
    )
    threshold = very_high_threshold(
        CLASS_BREAKS,
        f"{scenario}_susceptibility",
        area=area
    )

    local_table = rf_uncertainty_summary(
        sd_raster,
        frequency_raster,
        n_models=10,
        area=area,
        scenario=scenario,
        pilot=pilot,
        threshold=threshold,
        threshold_method="regional_p80_ensemble_mean"
    )
    local_tables.append(local_table)
    local_table.to_csv(out_dir / "local_summary.csv", index=False)

    very_high_tables.append(
        rf_uncertainty_very_high_summary(
            sd_raster,
            frequency_raster,
            class_raster,
            n_models=10,
            area=area,
            scenario=scenario,
            pilot=pilot,
            threshold=threshold,
            threshold_method="regional_p80_ensemble_mean"
        )
    )

summary_df = pd.concat(local_tables, ignore_index=True)
very_high_df = pd.concat(very_high_tables, ignore_index=True)
output = root / "rf_uncertainty.xlsx"

with pd.ExcelWriter(output) as writer:
    thresholds_df.to_excel(
        writer,
        sheet_name="Regional_thresholds",
        index=False
    )
    summary_df.to_excel(
        writer,
        sheet_name="Local_summary",
        index=False
    )
    very_high_df.to_excel(
        writer,
        sheet_name="Very_high_stability",
        index=False
    )

print("Resultados guardados em:", output)
very_high_df